# MÓDULO 4
## Tema 2. Manejo de archivos estructurados: lectura y escritura de CSV y JSON

**Objetivos:**
- Leer y escribir CSV y JSON con el estándar (`csv`, `json`) de forma robusta.
- Controlar separadores, comillas, cabeceras y tipos.
- Trabajar con JSON anidado y JSON Lines (muy usado en datos/logs).
- Preparar datos para análisis y para intercambio con APIs.

## Índice

- CSV con `csv` (lectura y escritura)
- Manejo de separadores, comillas y valores nulos
- JSON: conceptos básicos y estructuras
- Lectura y escritura de JSON con `json`
- Modificación y uso práctico de ficheros JSON
- Procesamiento de JSON (métricas y transformaciones)
- JSON Lines (NDJSON) para eventos y grandes volúmenes
- Errores comunes y buenas prácticas


## CSV con `csv`

El módulo `csv` es el estándar para lectura/escritura:

- `csv.reader` / `csv.writer` trabajan con listas.
- `csv.DictReader` / `csv.DictWriter` trabajan con diccionarios (recomendado en la mayoría de casos).

Caso realista: export de usuarios desde un CRM.

In [ ]:
import csv
from pathlib import Path

csv_path = Path("usuarios.csv")

filas = [
    {"id": 1, "email": "ana@example.com", "pais": "ES", "activo": "yes"},
    {"id": 2, "email": "luis@example.com", "pais": "ES", "activo": "no"},
    {"id": 3, "email": "marta@example.com", "pais": "PT", "activo": "yes"},
]

# Escribir CSV (DictWriter)
with csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "email", "pais", "activo"], delimiter=";")
    writer.writeheader()
    writer.writerows(filas)

print("CSV generado:", csv_path)

# Leer CSV (DictReader)
with csv_path.open("r", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f, delimiter=";")
    data = list(reader)

print(data[:2])

CSV generado: usuarios.csv
[{'id': '1', 'email': 'ana@example.com', 'pais': 'ES', 'activo': 'yes'}, {'id': '2', 'email': 'luis@example.com', 'pais': 'ES', 'activo': 'no'}]


## Separadores, comillas y valores nulos

En España es habitual encontrar CSV con `;` como separador.
También aparecen valores vacíos o con espacios.

Reglas prácticas:
- Si controlas tú el formato, usa `,` y UTF-8.
- Si recibes el fichero, inspecciona el separador y la codificación.

In [3]:
import csv
from pathlib import Path

csv_sc_path = Path("ventas_sc.csv")
texto = "id;importe;cliente\n1;19.99;Ana\n2;;José\n3;5.00;\n"
csv_sc_path.write_text(texto, encoding="utf-8")

with csv_sc_path.open("r", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f, delimiter=";")
    for row in reader:
        # normalización simple
        importe = row["importe"].strip() if row["importe"] else None
        cliente = row["cliente"].strip() if row["cliente"] else None
        print(row["id"], importe, cliente)

1 19.99 Ana
2 None José
3 5.00 None


## JSON con `json`

JSON es el formato más común para intercambio con APIs.

- `json.load(f)` / `json.dump(obj, f)`
- `json.loads(str)` / `json.dumps(obj)`

Caso realista: respuesta de una API de pedidos.

## JSON desde cero: estructura y tipos

JSON es **texto** con una estructura estándar. Antes de leer/escribir, conviene tener claro qué tipos existen y cómo se traducen a Python:

- `object` → `dict`
- `array` → `list`
- `string` → `str`
- `number` → `int` / `float`
- `boolean` → `bool` (`true`/`false` en JSON)
- `null` → `None`

Puntos importantes:
- Los strings en JSON usan **comillas dobles**.
- JSON no tiene fechas, sets, tuplas, ni objetos Python: hay que convertirlos (por ejemplo, fechas a ISO 8601).


In [ ]:
import json

texto_json = '''
{
  "nombre": "Ana",
  "edad": 29,
  "premium": true,
  "hobbies": ["correr", "leer"],
  "direccion": null,
  "scores": {"python": 8.5, "sql": 7}
}
'''

data = json.loads(texto_json)

print("Contenido:", data)
print("\nTipos en Python:")
print("data:", type(data))
print("data['hobbies']:", type(data["hobbies"]))
print("data['premium']:", type(data["premium"]))
print("data['direccion']:", type(data["direccion"]))
print("data['scores']:", type(data["scores"]))

Contenido: {'nombre': 'Ana', 'edad': 29, 'premium': True, 'hobbies': ['correr', 'leer'], 'direccion': None, 'scores': {'python': 8.5, 'sql': 7}}

Tipos en Python:
data: <class 'dict'>
data['hobbies']: <class 'list'>
data['premium']: <class 'bool'>
data['direccion']: <class 'NoneType'>
data['scores']: <class 'dict'>


In [5]:
import json
config = {
    "app": {"name": "MiApp", "version": "1.4.2"},
    "debug": True,
    "retries": 3,
    "timeout_s": 2.5,
    "features": {"beta": False, "new_ui": True},
    "allowed_countries": ["ES", "PT", "FR"],
}
print(type(config))
config_json = json.dumps(config)
print(type(config_json))
print(config)
print(config_json)

<class 'dict'>
<class 'str'>
{'app': {'name': 'MiApp', 'version': '1.4.2'}, 'debug': True, 'retries': 3, 'timeout_s': 2.5, 'features': {'beta': False, 'new_ui': True}, 'allowed_countries': ['ES', 'PT', 'FR']}
{"app": {"name": "MiApp", "version": "1.4.2"}, "debug": true, "retries": 3, "timeout_s": 2.5, "features": {"beta": false, "new_ui": true}, "allowed_countries": ["ES", "PT", "FR"]}


## `load/dump` vs `loads/dumps` (lo que más confunde al principio)

Son lo mismo, pero cambia **dónde está el JSON**:

- `json.loads(texto)` / `json.dumps(obj)` trabajan con **strings**.
- `json.load(f)` / `json.dump(obj, f)` trabajan con **archivos abiertos**.

En proyectos reales, suele ser más idiomático usar `load/dump` cuando ya estás trabajando con archivos.


In [ ]:
import json
from pathlib import Path

config_path = Path("config_app.json")
config_path_min = Path("config_app.min.json")

config = {
    "app": {"name": "MiApp", "version": "1.4.2"},
    "debug": True,
    "retries": 3,
    "timeout_s": 2.5,
    "features": {"beta": False, "new_ui": True},
    "allowed_countries": ["ES", "PT", "FR"],
}

# Escritura con dump (bonito)
with config_path.open("w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

# Escritura minificada (útil para transmitir/almacenar, menos legible)
with config_path_min.open("w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, separators=(",", ":"))

# Lectura con load
with config_path.open("r", encoding="utf-8") as f:
    config_leida = json.load(f)

print("Config leída:", config_leida["app"]["name"], config_leida["app"]["version"])
print("Debug:", config_leida["debug"])
print("Features:", config_leida["features"])

Config leída: MiApp 1.4.2
Debug: True
Features: {'beta': False, 'new_ui': True}


## Acceso seguro: datos incompletos (caso real)

En datos reales (APIs, logs, ficheros compartidos) es habitual que:
- falten claves
- haya valores `null`
- un campo cambie de nombre o de tipo

Patrones básicos:
- `dict.get("clave", default)` para evitar `KeyError`
- `...get("anidado", {})...` para navegar sin romper
- validaciones simples antes de usar un campo


In [3]:
import json
from pathlib import Path

usuarios_path = Path("usuarios.json")

usuarios = [
    {"email": "ana@example.com", "name": "Ana", "country": "ES"},
    {"email": "luis@example.com", "name": "Luis", "country": "ES", "phone": "+34 600 111 222"},
    {"email": "mai@example.com", "name": "Mai", "country": None},  # country desconocido
]

with usuarios_path.open("w", encoding="utf-8") as f:
    json.dump(usuarios, f, ensure_ascii=False, indent=2)

# Lectura y acceso seguro
with usuarios_path.open("r", encoding="utf-8") as f:
    usuarios_leidos = json.load(f)

for u in usuarios_leidos:
    email = u.get("email", "SIN_EMAIL")
    nombre = u.get("name", "SIN_NOMBRE")
    pais = u.get("country") or "DESCONOCIDO"   # cubre None o string vacío
    telefono = u.get("phone", "NO_DISPONIBLE") # clave opcional

    print(f"{nombre} ({email}) | país={pais} | teléfono={telefono}")

Ana (ana@example.com) | país=ES | teléfono=NO_DISPONIBLE
Luis (luis@example.com) | país=ES | teléfono=+34 600 111 222
Mai (mai@example.com) | país=DESCONOCIDO | teléfono=NO_DISPONIBLE


## Errores comunes: JSON inválido y `JSONDecodeError`

JSON tiene reglas estrictas. Dos muy típicas:
- Strings con **comillas simples** → inválido (en JSON deben ser dobles)
- Comas finales (trailing commas) → inválido

Cuando el texto no es JSON válido, `json.loads` lanza `json.JSONDecodeError`.


In [4]:
import json

ejemplos_invalidos = [
    "{'a': 1}",          # comillas simples
    '{"a": 1,}',         # coma final
    '{"a": True}',       # True/False (Python) en lugar de true/false (JSON)
]

for s in ejemplos_invalidos:
    try:
        json.loads(s)
    except json.JSONDecodeError as e:
        print("JSON inválido:", s)
        print("  ->", e)
        print()

JSON inválido: {'a': 1}
  -> Expecting property name enclosed in double quotes: line 1 column 2 (char 1)

JSON inválido: {"a": 1,}
  -> Illegal trailing comma before end of object: line 1 column 8 (char 7)

JSON inválido: {"a": True}
  -> Expecting value: line 1 column 7 (char 6)



## Transformación y reporte: “normalizar” datos y guardar resultados

Caso típico: recibes un JSON con estructura anidada y necesitas:
- calcular métricas
- quedarte con un resumen “plano”
- guardar un reporte para otra etapa del pipeline

Ejemplo: ventas del día (varios pedidos), generamos un reporte JSON con totales y top productos.


In [5]:
import json
from pathlib import Path
from collections import defaultdict

ventas_path = Path("ventas_dia.json")
reporte_path = Path("reporte_ventas.json")

ventas = [
    {
        "order_id": 1001,
        "customer": {"email": "ana@example.com", "country": "ES"},
        "items": [{"sku": "BOOK-001", "qty": 1, "price": 30.0}, {"sku": "COFFEE-002", "qty": 2, "price": 2.5}],
        "status": "paid",
    },
    {
        "order_id": 1002,
        "customer": {"email": "luis@example.com", "country": "ES"},
        "items": [{"sku": "BOOK-001", "qty": 2, "price": 30.0}],
        "status": "paid",
    },
    {
        "order_id": 1003,
        "customer": {"email": "mai@example.com", "country": "FR"},
        "items": [{"sku": "COFFEE-002", "qty": 1, "price": 2.5}],
        "status": "cancelled",
    },
]

# Guardamos el input (simula “respuesta API” o export)
with ventas_path.open("w", encoding="utf-8") as f:
    json.dump(ventas, f, ensure_ascii=False, indent=2)

# Cargamos y procesamos
with ventas_path.open("r", encoding="utf-8") as f:
    ventas_leidas = json.load(f)

conteo_por_status = defaultdict(int)
ingresos_totales = 0.0
unidades_por_sku = defaultdict(int)
ingresos_por_sku = defaultdict(float)

for pedido in ventas_leidas:
    status = pedido.get("status", "unknown")
    conteo_por_status[status] += 1

    # Solo contamos ingresos de pedidos pagados (caso realista)
    if status != "paid":
        continue

    for it in pedido.get("items", []):
        sku = it.get("sku", "SIN_SKU")
        qty = int(it.get("qty", 0))
        price = float(it.get("price", 0.0))
        subtotal = qty * price

        ingresos_totales += subtotal
        unidades_por_sku[sku] += qty
        ingresos_por_sku[sku] += subtotal

reporte = {
    "n_pedidos": len(ventas_leidas),
    "conteo_por_status": dict(conteo_por_status),
    "ingresos_totales_paid": round(ingresos_totales, 2),
    "unidades_por_sku": dict(unidades_por_sku)
}

with reporte_path.open("w", encoding="utf-8") as f:
    json.dump(reporte, f, ensure_ascii=False, indent=2)

print("Reporte generado:", reporte_path)
print(json.dumps(reporte, ensure_ascii=False, indent=2))

Reporte generado: reporte_ventas.json
{
  "n_pedidos": 3,
  "conteo_por_status": {
    "paid": 2,
    "cancelled": 1
  },
  "ingresos_totales_paid": 95.0,
  "unidades_por_sku": {
    "BOOK-001": 3,
    "COFFEE-002": 2
  }
}


## Actualizar un JSON en disco (leer → modificar → guardar)

Patrón típico con configuraciones:
1) leer `config.json`
2) cambiar valores
3) reescribir el fichero con `indent=2`

Esto aparece en scripts de despliegue, herramientas CLI y mantenimiento.


In [ ]:
import json
from pathlib import Path

config_path = Path("config_app.json")

# Leer
with config_path.open("r", encoding="utf-8") as f:
    config = json.load(f)

# Modificar
config["debug"] = False
config["retries"] = config.get("retries", 0) + 1
config.setdefault("features", {})
config["features"]["beta"] = True

# Guardar
with config_path.open("w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Config actualizada. debug =", config["debug"], "| retries =", config["retries"], "| beta =", config["features"]["beta"])

In [7]:
import json
from pathlib import Path

json_path = Path("pedido.json")
json_path_minificado = Path("pedido_minificado.json")

pedido = {
    "order_id": 123,
    "customer": {"email": "ana@example.com", "country": "ES"},
    "items": [
        {"sku": "BOOK-001", "qty": 1, "price": 30.0},
        {"sku": "COFFEE-002", "qty": 2, "price": 2.5},
    ],
    "status": "paid",
}

json_path.write_text(json.dumps(pedido, ensure_ascii=False, indent=2), encoding="utf-8")
json_path_minificado.write_text(json.dumps(pedido, ensure_ascii=False), encoding="utf-8")
print("JSON generado:", json_path)

JSON generado: pedido.json


In [8]:
# Leer el JSON
pedido = json.loads(json_path.read_text(encoding="utf-8"))

# Ver el contenido completo
print("Pedido completo:")
print(pedido)

print("\nDatos principales:")
print("Order ID:", pedido["order_id"])
print("Estado:", pedido["status"])
print("Email cliente:", pedido["customer"]["email"])
print("País:", pedido["customer"]["country"])

print("\nItems del pedido:")
total = 0.0
for item in pedido["items"]:
    subtotal = item["qty"] * item["price"]
    total += subtotal
    print(
        f"- SKU: {item['sku']} | "
        f"Cantidad: {item['qty']} | "
        f"Precio: {item['price']} € | "
        f"Subtotal: {subtotal} €"
    )

print("\nTotal del pedido:", total, "€")

Pedido completo:
{'order_id': 123, 'customer': {'email': 'ana@example.com', 'country': 'ES'}, 'items': [{'sku': 'BOOK-001', 'qty': 1, 'price': 30.0}, {'sku': 'COFFEE-002', 'qty': 2, 'price': 2.5}], 'status': 'paid'}

Datos principales:
Order ID: 123
Estado: paid
Email cliente: ana@example.com
País: ES

Items del pedido:
- SKU: BOOK-001 | Cantidad: 1 | Precio: 30.0 € | Subtotal: 30.0 €
- SKU: COFFEE-002 | Cantidad: 2 | Precio: 2.5 € | Subtotal: 5.0 €

Total del pedido: 35.0 €


## JSON Lines (NDJSON)

NDJSON = un JSON por línea. Es muy usado en:
- logs estructurados
- pipelines de datos
- export/import de grandes volúmenes

Ventaja: puedes procesar línea a línea sin cargar todo en memoria.

In [9]:
import json
from pathlib import Path

ndjson_path = Path("eventos.ndjson")

eventos = [
    {"ts": "2026-01-21T10:00:00", "type": "login", "user": "ana@example.com"},
    {"ts": "2026-01-21T10:05:00", "type": "purchase", "user": "ana@example.com", "amount": 19.99},
    {"ts": "2026-01-21T10:06:00", "type": "logout", "user": "ana@example.com"},
]

with ndjson_path.open("w", encoding="utf-8") as f:
    for e in eventos:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")

# Lectura streaming
conteo = {}
with ndjson_path.open("r", encoding="utf-8") as f:
    for line in f:
        e = json.loads(line)
        conteo[e["type"]] = conteo.get(e["type"], 0) + 1

print(conteo)

{'login': 1, 'purchase': 1, 'logout': 1}


## Trampas comunes y buenas prácticas

- Para CSV: usa `newline=""` en `open()` (evita líneas en blanco en Windows).
- Documenta el separador y el encoding si lo compartes con terceros.
- En JSON: usa `ensure_ascii=False` si quieres mantener tildes sin escapar.
- Para grandes volúmenes: NDJSON + procesado streaming.
- No evalúes JSON con `eval` (nunca). Usa `json`.
- JSON estricto: comillas dobles, sin comas finales; captura `json.JSONDecodeError` si el input viene de fuera.
- Si trabajas con archivos, prefiere `json.load/json.dump` (evitas `read_text`/`write_text` + `loads`/`dumps`).
- Si necesitas reproducibilidad (differences en Git), puedes usar `sort_keys=True` en `json.dump`.

## Mini-práctica

1) Genera un CSV de “productos” con: `sku`, `nombre`, `precio`, `stock`.  
2) Léelo con `DictReader` y calcula el valor total de inventario (`precio * stock`).  
3) Exporta a JSON un resumen con: número de productos, valor total, top 3 por valor.